# AXE 2 — 01. Build Interactions
_Notebook 1/2 — Unification Spotify + Netflix + YouTube → `data/warehouse/interactions.parquet`_

## Approche : Virtual Users (mois)
Avec un seul utilisateur réel, ALS PySpark échoue (matrice rang-1 non-définie).  
**Solution** : chaque mois d'activité = un virtual user.

- `user_id` = `year * 100 + month` (ex: 202401 = Janvier 2024)
- ~80 virtual users → ALS peut faire de vraie collaborative filtering
- Items co-consommés dans le même mois → similaires dans l'espace latent

**Output** : `data/warehouse/interactions.parquet`  
Colonnes : `user_id`, `item_id`, `item_title`, `platform`, `play_count`

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
# Si ce notebook plante avec "Connection refused" ou "no resources" :
#   → Kernel > Restart Kernel and Clear Outputs, puis relancer depuis ici.

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Interactions") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"App : {spark.sparkContext.appName}")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.parquet(os.path.join(WAREHOUSE, name))

Spark version : 3.5.5
App : PySparkShell
Warehouse: /opt/spark/warehouse


26/04/06 14:26:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# ── 1. SPOTIFY STREAMS ────────────────────────────────────────────────────────
# Virtual user = year * 100 + month
# listen_year et listen_month sont déjà dans le parquet

spotify_raw = read_table("spotify_streams")
print(f"Spotify brut : {spotify_raw.count():,} streams")
spotify_raw.printSchema()

spotify = spotify_raw.select(
    F.concat_ws(" — ", F.col("artistName"), F.col("trackName")).alias("item_title"),
    F.lit("spotify").alias("platform"),
    # user_id = year * 100 + month (ex: 202401)
    (F.col("listen_year") * 100 + F.month(F.col("listen_ts"))).cast(IntegerType()).alias("user_id")
).filter(
    F.col("item_title").isNotNull() &
    (F.trim(F.col("item_title")) != "") &
    (F.col("item_title") != " — ") &
    F.col("user_id").isNotNull()
)

print(f"Spotify filtré : {spotify.count():,}")

Spotify brut : 33,972 streams
root
 |-- artistName: string (nullable = true)
 |-- trackName: string (nullable = true)
 |-- msPlayed: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- listen_ts: timestamp (nullable = true)
 |-- listen_year: integer (nullable = true)
 |-- listen_month: string (nullable = true)
 |-- listen_hour: integer (nullable = true)
 |-- listen_weekday: integer (nullable = true)
 |-- listen_week: integer (nullable = true)
 |-- is_night: boolean (nullable = true)
 |-- interaction_weight: double (nullable = true)

Spotify filtré : 33,972


In [3]:
# ── 2. NETFLIX VIEWS ──────────────────────────────────────────────────────────
# watch_year disponible ; watch_month est au format 'yyyy-MM' → extraire le numéro

netflix_raw = read_table("netflix_views")
print(f"Netflix brut : {netflix_raw.count():,} views")

netflix = netflix_raw.select(
    F.col("show_title").alias("item_title"),
    F.lit("netflix").alias("platform"),
    (F.col("watch_year") * 100 + F.month(F.col("watch_date"))).cast(IntegerType()).alias("user_id")
).filter(
    F.col("item_title").isNotNull() &
    (F.trim(F.col("item_title")) != "") &
    F.col("user_id").isNotNull()
)

print(f"Netflix filtré : {netflix.count():,}")

Netflix brut : 4,288 views
Netflix filtré : 4,240


In [4]:
# ── 3. YOUTUBE — EXCLU ────────────────────────────────────────────────────────
# YouTube retiré des interactions : trop de bruit (pubs, vidéos courtes, auto-play)
# et les titres YouTube apportent peu de valeur sémantique pour la recommandation.
# Sources retenues : Spotify (signal fort msPlayed) + Netflix (contenu long format).
print("YouTube exclu des interactions (trop de bruit).")

YouTube exclu des interactions (trop de bruit).


In [5]:
# ── 4. UNION + NORMALISATION ──────────────────────────────────────────────────
# Sources : Spotify + Netflix uniquement
raw_all = spotify.union(netflix)

# On calcule le poids total par plateforme pour rééquilibrer
platform_counts = raw_all.groupBy("platform").count().collect()
counts_dict = {row['platform']: row['count'] for row in platform_counts}
print(f"Événements bruts : {counts_dict}")

# Facteur d'échelle : on veut que chaque plateforme pèse autant
max_count = max(counts_dict.values())
scaling = {p: max_count / c for p, c in counts_dict.items()}
print(f"Facteurs de normalisation : {scaling}")

from pyspark.sql.functions import col, create_map, lit
from itertools import chain

mapping_expr = create_map([lit(x) for x in chain(*scaling.items())])
raw_normalized = raw_all.withColumn("weight", mapping_expr.getItem(col("platform")))

# .cache() : interactions_agg est utilisé 2x (stats + filtre noise)
interactions_agg = raw_normalized.groupBy("user_id", "item_title", "platform").agg(
    F.sum("weight").alias("play_count")
).cache()

n_users    = interactions_agg.select("user_id").distinct().count()
n_items_raw = interactions_agg.select("item_title").distinct().count()
print(f"Virtual users (mois) : {n_users}")
print(f"Items distincts (brut) : {n_items_raw:,}")

interactions_agg.groupBy("platform").agg(
    F.countDistinct("item_title").alias("n_items"),
    F.sum("play_count").alias("total_weight")
).orderBy("platform").show()

/opt/spark/python/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


Événements bruts : {'spotify': 33972, 'netflix': 4240}
Facteurs de normalisation : {'spotify': 1.0, 'netflix': 8.012264150943397}
Virtual users (mois) : 83
Items distincts (brut) : 7,986
+--------+-------+-----------------+
|platform|n_items|     total_weight|
+--------+-------+-----------------+
| netflix|    638|33972.00000000032|
| spotify|   7348|          33972.0|
+--------+-------+-----------------+



In [6]:
# ── 5. FILTRE BRUIT ───────────────────────────────────────────────────────────
# Exclure les items vus dans < 2 mois distincts (bruit)
# Un item vu plusieurs mois = signal plus fiable

item_month_count = interactions_agg.groupBy("item_title").agg(
    F.countDistinct("user_id").alias("n_months_seen")
).filter(F.col("n_months_seen") >= 2)

interactions_filtered = interactions_agg.join(item_month_count.select("item_title"), on="item_title", how="inner")

n_filtered = interactions_filtered.select("item_title").distinct().count()
print(f"Items après filtre (vus dans >= 2 mois) : {n_filtered:,}")
print(f"Interactions : {interactions_filtered.count():,}")

Items après filtre (vus dans >= 2 mois) : 3,884
Interactions : 15,620


In [7]:
# ── 6. STRING INDEXER → item_id entier ────────────────────────────────────────
# ALS nécessite des IDs entiers

from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="item_title", outputCol="item_id_float", handleInvalid="keep")
indexer_model = indexer.fit(interactions_filtered)
interactions_indexed = indexer_model.transform(interactions_filtered) \
    .withColumn("item_id", F.col("item_id_float").cast(IntegerType())) \
    .drop("item_id_float")

n_items = interactions_indexed.select("item_id").distinct().count()
print(f"Items indexés : {n_items:,}")
interactions_indexed.orderBy(F.desc("play_count")).show(10, truncate=50)

Items indexés : 3,884
+----------------------+-------+--------+------------------+-------+
|            item_title|user_id|platform|        play_count|item_id|
+----------------------+-------+--------+------------------+-------+
|      Naruto Shippuden| 202303| netflix| 1330.035849056605|    270|
|Hunter X Hunter (2011)| 202501| netflix|1169.7905660377382|   2140|
|      Naruto Shippuden| 202302| netflix| 1137.741509433965|    270|
|                Naruto| 202302| netflix|1113.7047169811349|   2374|
| The Seven Deadly Sins| 202505| netflix| 705.0792452830202|   2559|
|          Regular Show| 202402| netflix| 624.9566037735857|   3588|
|            Fairy Tail| 202007| netflix| 608.9320754716988|   1441|
|      Naruto Shippuden| 201909| netflix| 520.7971698113208|    270|
|            Fairy Tail| 202006| netflix| 520.7971698113208|   1441|
|                Naruto| 201906| netflix|480.73584905660374|   2374|
+----------------------+-------+--------+------------------+-------+
only showing

In [8]:
# ── 7. ÉCRITURE warehouse/interactions ────────────────────────────────────────

out_df = interactions_indexed.select(
    "user_id", "item_id", "item_title", "platform", "play_count"
)

out_path = os.path.join(WAREHOUSE, "interactions")
out_df.write.mode("overwrite").parquet(out_path)

print(f"Écrit : {out_path}")

# Vérification
check = spark.read.parquet(out_path)
print(f"Lignes : {check.count():,}")
check.printSchema()
check.orderBy(F.desc("play_count")).show(10, truncate=50)

spark.stop()
print("Notebook 01 terminé. Lance 02_als_model.ipynb.")

Écrit : /opt/spark/warehouse/interactions
Lignes : 15,620
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- item_title: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- play_count: double (nullable = true)

+-------+-------+----------------------+--------+------------------+
|user_id|item_id|            item_title|platform|        play_count|
+-------+-------+----------------------+--------+------------------+
| 202303|    270|      Naruto Shippuden| netflix| 1330.035849056605|
| 202501|   2140|Hunter X Hunter (2011)| netflix|1169.7905660377382|
| 202302|    270|      Naruto Shippuden| netflix| 1137.741509433965|
| 202302|   2374|                Naruto| netflix|1113.7047169811349|
| 202505|   2559| The Seven Deadly Sins| netflix| 705.0792452830202|
| 202402|   3588|          Regular Show| netflix| 624.9566037735857|
| 202007|   1441|            Fairy Tail| netflix| 608.9320754716988|
| 201909|    270|      Naruto Shippuden| n